# Calculation of molecular descriptors


This stage focuses on preparing experimental earthworm acute toxicity data for QSAR classification modeling.

The workflow includes:

- Selection of toxicity data based on OECD Guideline 207.
- Harmonization of experimental toxicity labels into binary classes (toxic = 1; non-toxic = 0).
- Chemical structure validation and SMILES standardization using RDKit.
- Removal of selected salts, mixtures, inorganic compounds, and duplicate chemical structures.
- Export of the curated dataset for subsequent molecular descriptor calculation and machine learning model development.

**Tools:** Python, Pandas, RDKit.

**Output:** `Earthworm_acute_toxicity.csv` containing curated SMILES and classification labels.

In [ ]:
#@title Installing the required packages
!pip install rdkit==2023.9.1
!pip install mordred==1.2.0
!pip install pandas --upgrade --force-reinstall # Reinstall pandas to ensure compatibility with the installed numpy

  Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six-1.17.0
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: python-dat

Finally, you can use the code below to specify where you would like to store and read your files. In this case, we will verify that it works using the file `Earthworm_acute_toxicity.csv`, which was created in the previous lesson.







In [ ]:
#@title Files I/O configuration
import os

## Modify and uncomment the appropiate lines below to mark the right path to the folder where all the files are going to be stored

# if you upload your files directly, decomment next line:
PATH = "/content/"

# if you use GoogleDrive, decomment next lines (You will need to give permission to Colab to access your Drive account):
# from google.colab import drive
# drive.mount('/content/drive')
# PATH = "/content/drive/MyDrive/PQSAR_Python_Course/" # Here we added a folder name where all the data is going to be stored in the google drive

## Upload the file Earthworm_acute_toxicity.csv to your folder and check whether it exists:
os.path.isfile(os.path.join(PATH, "sample_data/Earthworm_acute_toxicity_raw_data.csv"))

True

## Section 1: Preparing the database

In this lesson, we will learn how to calculate molecular descriptors for a series of chemical substances. Therefore, the first step is to load the list of substances (the dataset) that will be used to build the model. In this case, we will use the curated dataset prepared in the previous lesson.

Import the Earthworm_acute_toxicity dataset after sanitisation (`Earthworm_acute_toxicity.csv`) and extract the `SMILES` column as a list.

Remember that this is a new Jupyter Notebook, so you will need to import any modules and libraries—such as `pandas`—even if they were used in Part 1.


In [ ]:
#@title Database importation

#Fill in the gaps to complete the code cells
import pandas as pd       # Import pandas

df_classification_final = pd.read_csv(os.path.join(PATH, "sample_data/Earthworm_acute_toxicity_raw_data.csv"), sep=';')   # The DataFrame containing the curated dataset

smiles_list = df_classification_final['SMILES'].tolist()    # This should be a list containing the SMILES strings

print(smiles_list) # Print to verify that it worked

['COc1nc(C)nc(NC(=O)NS(=O)(=O)c2ccccc2OCCCl)n1', 'CCO/N=C(\\CC)C1=C(O)CC(c2c(C)cc(C)cc2C)CC1=O', 'CC1(C(=O)Nc2ccc(O)c(Cl)c2Cl)CCCCC1', 'O=C(NC(=O)c1c(F)cccc1F)Nc1cc(Cl)c(OC(F)(F)C(F)C(F)(F)F)cc1Cl', 'O=[N+]([O-])c1cc(C(F)(F)F)c(Cl)c([N+](=O)[O-])c1Nc1ncc(C(F)(F)F)cc1Cl', 'Oc1ccccc1-c1ccccc1', 'CC1(C)CC/C(=C/c2ccc(Cl)cc2)C1(O)Cn1cncn1', 'CCS(=O)CCSP(=O)(OC)OC', 'CCC(=O)Nc1ccc(Cl)c(Cl)c1', 'O=C(Nc1ccccc1)N(Cc1ccc(Cl)cc1)C1CCCC1', 'CCCCOC(=O)C(C)Oc1ccc(Oc2ccc(C(F)(F)F)cn2)cc1', '[SH-].[SH-].[Fe].[Fe+2]', 'CC1(c2ccc(Oc3ccccc3)cc2)OC(=O)N(Nc2ccccc2)C1=O', 'Cc1nn(C)c(F)c1C(=O)Nc1ccccc1C(C)CC(C)C', 'Oc1nc2ccc(Cl)cc2nc1O', 'CCCSP(=O)(OCC)SCCC', 'CCOC(=O)[C@@H](C)Oc1ccc(Oc2cnc3cc(Cl)ccc3n2)cc1', 'O=C(NC(=O)c1c(F)cccc1F)Nc1ccc(Cl)cc1', 'Nc1c(Cl)c(F)nc(OCC(=O)O)c1Cl', 'O=C(O)CNCP(=O)(O)O', 'Cc1cc(C(F)(C(F)(F)F)C(F)(F)F)ccc1NC(=O)c1cccc(I)c1C(=O)NC(C)(C)CS(C)(=O)=O', '[Na+].CCCCCCCCCCCCC1=C(OC(C)=O)C(=O)c2ccccc2C1=O', 'Cc1cc(C)c(C2=C(OC(=O)CC(C)(C)C)C3(CCCC3)OC2=O)c(C)c1', 'Cc1cc(C)nc(NC(=O)NS(=O)

Let’s verify that the file is correct. Count the number of molecules and ensure that both the `SMILES` and `y` columns are present.

In [ ]:
#@title Verifying the dataset

#Print the column names
print(df_classification_final.columns)

#Print the length of the Dataframe
print(len(df_classification_final))

Index(['SMILES', 'value', 'Experimental guidelines'], dtype='str')
202



## Section 2: Manual Calculation of Selected Descriptors

In this lesson, we will learn how to calculate molecular descriptors. We will begin by calculating a few descriptors manually and in greater detail. Later, we will use external libraries to extract a broader range of molecular descriptors from the `rdkit` and `mordred` modules in a more efficient and streamlined manner.

We will now calculate several molecular descriptors, including molecular weight, the number of hydrogen bond acceptors, the number of hydrogen bond donors, the number of heteroatoms, and the number of rotatable bonds. To begin, we will calculate the molecular weight for a single compound.

We will be using descriptor functions from the rdkit.Chem module. Therefore, the first step is to import them. For further information, consult the following link:
https://www.rdkit.org/docs/source/rdkit.Chem.Descriptors.html




In [ ]:
#@title Import the rdkit.Chem and the Descriptors modules
from rdkit import Chem
from rdkit.Chem import Descriptors

From this module, we will use the MolWt descriptor. As an initial example, we will examine the molecular weight of aspirin. You can find information about its chemical structure, including the SMILES representation, from various sources such as:

https://pubchem.ncbi.nlm.nih.gov/compound/Aspirin

https://en.wikipedia.org/wiki/Aspirin

In [ ]:
#@title Calculate the Molecular weight of Aspirin

#Introduce the Aspirin SMILES
smi = 'CC(=O)Oc1ccccc1C(=O)O'
#Enter the SMILES string for Aspirin
mol = Chem.MolFromSmiles(smi)

#Convert the SMILES string to a mol object
mol_wt = Descriptors.MolWt(mol)


print(mol_wt)

180.15899999999996


### **Q1.** What is the molecular weight of the Aspirin?

Enter the value on the training platform to assess your progress.


As we will be working with a dataset, it is common practice not to calculate the value of a descriptor for a single compound, but rather to perform multiple calculations across the dataset.

To achieve this, we will use loops to iterate over both the descriptors and the molecules. In this example, we will begin by creating an empty list to store the values of each descriptor.

We will then iterate through the list of SMILES from our DataFrame. For each molecule, we will calculate the descriptor values and store the results in the corresponding lists.

Let us begin with a simple example using the descriptor we have already introduced: **molecular weight**.

In [ ]:
#@title  Molecular weight of the dataset


# Create an empty list to store the molecular weight values
molwt_list = []

for sm in smiles_list:  #Iterate through the list of SMILES obtained in the “Database Importation” cell
    # Use RDKit to read the SMILES string
    mol = Chem.MolFromSmiles(sm)

    # Calculate the molecular weight of the molecule
    if mol is not None:
        mol_wt = Descriptors.MolWt(mol)
    else:
        mol_wt = float('nan') # Handle unparseable SMILES

    # Append the value to the list
    molwt_list.append(mol_wt)

#Retain this code to print the list
print(molwt_list)

[401.83200000000016, 329.4400000000001, 302.20100000000014, 511.15200000000004, 465.0930000000001, 170.211, 317.82, 246.28999999999996, 218.08299999999997, 328.84300000000013, 383.366, 177.83999999999997, 374.3960000000001, 317.40800000000013, 196.593, 242.34599999999995, 372.8080000000001, 310.687, 255.03199999999998, 169.073, 682.396, 407.50600000000014, 370.4890000000002, 406.42000000000024, 381.3700000000001, 376.1780000000001, 387.86300000000006, 336.6460000000001, 347.2640000000001, 202.21699999999998, 376.67100000000005, 487.3480000000002, 257.437, 337.5899999999999, 321.37600000000003, 198.182, 162.214, 296.33200000000005, 360.413, 440.7380000000002, 225.25500000000002, 209.2, 325.4740000000001, 271.36, 359.28900000000004, 372.20600000000013, 305.4470000000001, nan, 307.825, 372.1490000000001, 238.20000000000002, 316.16200000000003, nan, 296.562, 264.668, 396.718, 491.12000000000006, 485.4530000000002, 312.373, 361.70300000000003, 304.35200000000003, nan, 235.13600000000002, 38

[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] Explicit valence for atom # 4 F, 2, is greater than permitted
[12:37:15] Explicit valence for atom # 1 Cl, 2, is greater than permitted
[12:37:15] Explicit valence for atom # 3 O, 4, is greater than permitted
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] non-ring atom 0 marked aromatic
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] SMILES Parse Error: syntax error while parsing: CO!OO=C
[12:37:15] SMILES Parse Error: Failed parsing SMILES 'CO!OO=C' for input: 'CO!OO=C'
[

We will now repeat the process using multiple descriptors. Refer to the RDKit documentation to find the functions for calculating the number of hydrogen bond acceptors, the number of hydrogen bond donors, the number of heteroatoms, and the number of rotatable bonds.

Then, extend the loop by creating a separate list for each descriptor and performing the corresponding calculations.


In [ ]:
#@title Other descriptors

# Create empty lists for storing the descriptor values
h_acc = [] # To store the number of hydrogen acceptors.
h_don = [] # To store the number of hydrogen donors
hetero = [] # To store the number of heteroatoms
rot = [] # To store the number of rotatable bonds


for sm in smiles_list:
    # Use RDKit to convert the SMILES string into a mol object
    mol = Chem.MolFromSmiles(sm)

    if mol is not None:
        # Calculate the number of hydrogen acceptors using the Descriptors module and append the value to the list
        h_acc.append(Descriptors.NumHAcceptors(mol))

        # Calculate the number of hydrogen donors using the Descriptors module and append the value to the list
        h_don.append(Descriptors.NumHDonors(mol))

        # Calculate the number of heteroatoms using the Descriptors module and append the value to the list
        hetero.append(Descriptors.NumHeteroatoms(mol))

        # Calculate the number of rotatable bonds using the Descriptors module and append the value to the list
        rot.append(Descriptors.NumRotatableBonds(mol))
    else:
        h_acc.append(float('nan'))
        h_don.append(float('nan'))
        hetero.append(float('nan'))
        rot.append(float('nan'))

#Retain this step to verify that all the lists have been populated with the corresponding descriptor values
print(len(h_acc),len(h_don),len(hetero),len(rot))

202 202 202 202


[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] Explicit valence for atom # 4 F, 2, is greater than permitted
[12:37:15] Explicit valence for atom # 1 Cl, 2, is greater than permitted
[12:37:15] Explicit valence for atom # 3 O, 4, is greater than permitted
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] non-ring atom 0 marked aromatic
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] SMILES Parse Error: syntax error while parsing: CO!OO=C
[12:37:15] SMILES Parse Error: Failed parsing SMILES 'CO!OO=C' for input: 'CO!OO=C'
[

We will now create a DataFrame to combine all the calculated descriptors as separate columns, including a column for the corresponding SMILES strings.

In [ ]:
#@title Combining the descriptors into a DataFrame
earthworm_table = pd.DataFrame({'SMILES': smiles_list,
                                'MolWt': molwt_list,
                                'NumHAcceptors': h_acc,
                                'NumHDonors': h_don,
                                'NumHeteroatoms': hetero,
                                'NumRotatableBonds': rot})
print(earthworm_table.head())

                                              SMILES    MolWt  NumHAcceptors  \
0       COc1nc(C)nc(NC(=O)NS(=O)(=O)c2ccccc2OCCCl)n1  401.832            8.0   
1        CCO/N=C(\CC)C1=C(O)CC(c2c(C)cc(C)cc2C)CC1=O  329.440            4.0   
2                 CC1(C(=O)Nc2ccc(O)c(Cl)c2Cl)CCCCC1  302.201            2.0   
3  O=C(NC(=O)c1c(F)cccc1F)Nc1cc(Cl)c(OC(F)(F)C(F)...  511.152            3.0   
4  O=[N+]([O-])c1cc(C(F)(F)F)c(Cl)c([N+](=O)[O-])...  465.093            6.0   

   NumHDonors  NumHeteroatoms  NumRotatableBonds  
0         2.0            12.0                7.0  
1         1.0             4.0                5.0  
2         2.0             5.0                2.0  
3         2.0            15.0                5.0  
4         1.0            16.0                4.0  


## Section 3: Calculating RDKit descriptors for the dataset




In the examples above, we calculated a few RDKit descriptors individually. However, it is more practical to calculate them collectively. This can be achieved using the `CalcMolDescriptors` function. As you will see, the results are returned in the form of a dictionary.



In [ ]:
#@title Example: Calculating all RDKit descriptors for a single molecule

smile = 'CCCCF'
mol = Chem.MolFromSmiles(smile)
descriptors = Descriptors.CalcMolDescriptors(mol)
print(descriptors)


{'MaxAbsEStateIndex': 10.993472222222222, 'MaxEStateIndex': 10.993472222222222, 'MinAbsEStateIndex': 0.15625, 'MinEStateIndex': -0.15625, 'qed': 0.4697630483856065, 'SPS': 8.4, 'MolWt': 76.11399999999999, 'HeavyAtomMolWt': 67.042, 'ExactMolWt': 76.068828508, 'NumValenceElectrons': 32, 'NumRadicalElectrons': 0, 'MaxPartialCharge': 0.08939119357686791, 'MinPartialCharge': -0.2511087929571838, 'MaxAbsPartialCharge': 0.2511087929571838, 'MinAbsPartialCharge': 0.08939119357686791, 'FpDensityMorgan1': 1.6, 'FpDensityMorgan2': 2.2, 'FpDensityMorgan3': 2.2, 'BCUT2D_MWHI': 19.141171173123897, 'BCUT2D_MWLOW': 10.3764777942596, 'BCUT2D_CHGHI': 1.7113535197379264, 'BCUT2D_CHGLO': -1.7603267324539424, 'BCUT2D_LOGPHI': 1.8267986654556, 'BCUT2D_LOGPLOW': -1.6659796205056847, 'BCUT2D_MRHI': 4.2347085588995315, 'BCUT2D_MRLOW': 0.4689259011960937, 'AvgIpc': 1.4056390622295665, 'BalabanJ': 2.19060968716425, 'BertzCT': 11.119415478763749, 'Chi0': 4.121320343559642, 'Chi0n': 3.4992848165688697, 'Chi0v': 3.

###**Q2** How many different descriptors are calculated?


In [ ]:
print(len(descriptors))

210


We will now apply this to the full dataset using a similar approach to the one above. In this case, as the results for all descriptors are returned as a dictionary, we will use a single list to store them.

Later, the list of dictionaries can be easily converted into a DataFrame; however, this will not include the SMILES strings. Therefore, we will concatenate it with the original DataFrame (`df_classification_final`) and name the resulting DataFrame `df_class_rdkit`.

In [ ]:
#@title Calculating all RDKit descriptors for the dataset

#Create a single list, as all descriptors are returned together in one dictionary
descriptors_list = []

for sm in smiles_list:
    # Use RDKit to convert the SMILES string into a mol object
    mol = Chem.MolFromSmiles(sm)

    # Calculate all descriptors using CalcMolDescriptors
    if mol is not None:
        descriptors_rdkit = Descriptors.CalcMolDescriptors(mol)
    else:
        # If SMILES is invalid, append an empty dictionary so that pd.DataFrame fills with NaN
        descriptors_rdkit = {}

    descriptors_list.append(descriptors_rdkit)

#Convert the list of descriptors into a DataFrame
df_rdkit_desc = pd.DataFrame(descriptors_list)

#Concatenate the descriptors DataFrame with the original DataFrame
df_class_rdkit = pd.concat([df_classification_final, df_rdkit_desc], axis=1)


#Retain this step to verify the resulting DataFrame
print(df_class_rdkit.shape)
print(df_class_rdkit.head())

[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] WARNING: not removing hydrogen atom without neighbors
[12:37:15] Explicit valence for atom # 4 F, 2, is greater than permitted
[12:37:16] Explicit valence for atom # 1 Cl, 2, is greater than permitted
[12:37:16] Explicit valence for atom # 3 O, 4, is greater than permitted


(202, 213)
                                              SMILES      value  \
0       COc1nc(C)nc(NC(=O)NS(=O)(=O)c2ccccc2OCCCl)n1  Non-toxic   
1        CCO/N=C(\CC)C1=C(O)CC(c2c(C)cc(C)cc2C)CC1=O   Positive   
2                 CC1(C(=O)Nc2ccc(O)c(Cl)c2Cl)CCCCC1  Non-toxic   
3  O=C(NC(=O)c1c(F)cccc1F)Nc1cc(Cl)c(OC(F)(F)C(F)...  non-toxic   
4  O=[N+]([O-])c1cc(C(F)(F)F)c(Cl)c([N+](=O)[O-])...   negative   

                             Experimental guidelines  MaxAbsEStateIndex  \
0  OECD Guidelines 207: Earthworm, Acute Toxicity...          12.444044   
1  OECD Guidelines 207: Earthworm, Acute Toxicity...          12.687206   
2  OECD Guidelines 207: Earthworm, Acute Toxicity...          12.362376   
3  OECD Guidelines 207: Earthworm, Acute Toxicity...          13.562079   
4  OECD Guidelines 207: Earthworm, Acute Toxicity...          13.024378   

   MaxEStateIndex  MinAbsEStateIndex  MinEStateIndex       qed        SPS  \
0       12.444044           0.018077       -4.199046  0.66

[12:37:18] Explicit valence for atom # 1 Cl, 2, is greater than permitted
[12:37:18] non-ring atom 0 marked aromatic


## Section 4: Calculating Mordred descriptors

Another widely used Python package for calculating molecular descriptors is Mordred. Further information can be found in the following publication:

https://jcheminf.biomedcentral.com/articles/10.1186/s13321-018-0258-y

and in the official documentation:

http://mordred-descriptor.github.io/documentation/master/

Let us examine how many descriptors are calculated by Mordred.

First, import `Calculator` and `descriptors` from the `mordred` library. Then, create a `Calculator` object (which we will label as `calc`), to be used for calculating the descriptors. By default, all available descriptors are included; however, options such as `ignore_3D` can be used to adjust the selection.


In [ ]:
#@title Preparing the mordred Calculator
# We need to import both the calculator and the list of descriptors
from mordred import Calculator, descriptors

# When creating the calculator, we will include the 3D descriptors by setting the parameter ignore_3D = False. If you wish to exclude them, you can set ignore_3D = True

calc = Calculator(descriptors, ignore_3D = False)

#Use this to inspect the calculated descriptors
print(calc.descriptors)
len(calc.descriptors)

(mordred.ABCIndex.ABCIndex(), mordred.ABCIndex.ABCGGIndex(), mordred.AcidBase.AcidicGroupCount(), mordred.AcidBase.BasicGroupCount(), mordred.AdjacencyMatrix.AdjacencyMatrix('SpAbs'), mordred.AdjacencyMatrix.AdjacencyMatrix('SpMax'), mordred.AdjacencyMatrix.AdjacencyMatrix('SpDiam'), mordred.AdjacencyMatrix.AdjacencyMatrix('SpAD'), mordred.AdjacencyMatrix.AdjacencyMatrix('SpMAD'), mordred.AdjacencyMatrix.AdjacencyMatrix('LogEE'), mordred.AdjacencyMatrix.AdjacencyMatrix('VE1'), mordred.AdjacencyMatrix.AdjacencyMatrix('VE2'), mordred.AdjacencyMatrix.AdjacencyMatrix('VE3'), mordred.AdjacencyMatrix.AdjacencyMatrix('VR1'), mordred.AdjacencyMatrix.AdjacencyMatrix('VR2'), mordred.AdjacencyMatrix.AdjacencyMatrix('VR3'), mordred.Aromatic.AromaticAtomsCount(), mordred.Aromatic.AromaticBondsCount(), mordred.AtomCount.AtomCount('Atom'), mordred.AtomCount.AtomCount('HeavyAtom'), mordred.AtomCount.AtomCount('Spiro'), mordred.AtomCount.AtomCount('Bridgehead'), mordred.AtomCount.AtomCount('Hetero'), m

1826

The input for the Mordred calculator consists of RDKit `mol` objects, so we must first create a list of `mol` objects. To do this, iterate through the list of SMILES strings and convert them into a list of `mol` objects (this can be done using a list comprehension).

In [ ]:
#@title Create a list of mol objects
mols = [Chem.MolFromSmiles(smi) for smi in smiles_list]

print(len(mols))

202


[12:37:18] WARNING: not removing hydrogen atom without neighbors
[12:37:18] WARNING: not removing hydrogen atom without neighbors
[12:37:18] WARNING: not removing hydrogen atom without neighbors
[12:37:18] WARNING: not removing hydrogen atom without neighbors
[12:37:18] Explicit valence for atom # 4 F, 2, is greater than permitted
[12:37:18] Explicit valence for atom # 1 Cl, 2, is greater than permitted
[12:37:18] Explicit valence for atom # 3 O, 4, is greater than permitted
[12:37:18] WARNING: not removing hydrogen atom without neighbors
[12:37:18] WARNING: not removing hydrogen atom without neighbors
[12:37:18] WARNING: not removing hydrogen atom without neighbors
[12:37:18] non-ring atom 0 marked aromatic
[12:37:18] WARNING: not removing hydrogen atom without neighbors
[12:37:18] WARNING: not removing hydrogen atom without neighbors
[12:37:18] SMILES Parse Error: syntax error while parsing: CO!OO=C
[12:37:18] SMILES Parse Error: Failed parsing SMILES 'CO!OO=C' for input: 'CO!OO=C'
[

The Mordred calculator includes a method called `pandas`, which calculates descriptors for an entire list of `mol` objects and returns the results as a DataFrame. This method should be called on the `calc` object created earlier (as `calc.pandas()`), with the list of `mol` objects as its input.

In [ ]:
#@title Calculate the Mordred descriptors

# Filter out None values from the mols list
valid_mols = [mol for mol in mols if mol is not None]

df_calc_mordred = calc.pandas(valid_mols)
print(df_calc_mordred)

 10%|▉         | 19/194 [00:02<00:19,  8.86it/s][12:37:21] WARNING: not removing hydrogen atom without neighbors
[12:37:21] WARNING: not removing hydrogen atom without neighbors
[12:37:21] WARNING: not removing hydrogen atom without neighbors
[12:37:21] WARNING: not removing hydrogen atom without neighbors
[12:37:21] WARNING: not removing hydrogen atom without neighbors
[12:37:21] WARNING: not removing hydrogen atom without neighbors
 13%|█▎        | 25/194 [00:03<00:24,  7.04it/s][12:37:22] WARNING: not removing hydrogen atom without neighbors
[12:37:22] WARNING: not removing hydrogen atom without neighbors
[12:37:22] WARNING: not removing hydrogen atom without neighbors
[12:37:22] WARNING: not removing hydrogen atom without neighbors
[12:37:22] WARNING: not removing hydrogen atom without neighbors
[12:37:22] WARNING: not removing hydrogen atom without neighbors
 23%|██▎       | 45/194 [00:07<00:27,  5.37it/s][12:37:25] WARNING: not removing hydrogen atom without neighbors
[12:37:25] 

/usr/local/lib/python3.12/dist-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce


 99%|█████████▉| 192/194 [00:40<00:00,  3.80it/s]

/usr/local/lib/python3.12/dist-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce


100%|██████████| 194/194 [00:40<00:00,  4.77it/s]

                                                   ABC  \
0    module 'numpy' has no attribute 'float'.\n`np....   
1    module 'numpy' has no attribute 'float'.\n`np....   
2    module 'numpy' has no attribute 'float'.\n`np....   
3    module 'numpy' has no attribute 'float'.\n`np....   
4    module 'numpy' has no attribute 'float'.\n`np....   
..                                                 ...   
189  module 'numpy' has no attribute 'float'.\n`np....   
190  module 'numpy' has no attribute 'float'.\n`np....   
191  module 'numpy' has no attribute 'float'.\n`np....   
192  module 'numpy' has no attribute 'float'.\n`np....   
193  module 'numpy' has no attribute 'float'.\n`np....   

                                                 ABCGG  nAcid  nBase  \
0    module 'numpy' has no attribute 'float'.\n`np....      0      0   
1    module 'numpy' has no attribute 'float'.\n`np....      0      0   
2    module 'numpy' has no attribute 'float'.\n`np....      0      0   
3    module 'nu

Once again, this DataFrame does not contain the SMILES strings, so we will concatenate it with the original DataFrame to include them. The resulting DataFrame will be named `df_class_mordred`.

In [ ]:
#@title Concatenate the two DataFrames

# Helper function to clean column names (copied here to ensure availability for Mordred DataFrame cleaning)
import ast # Import the ast module for literal_eval
def clean_col_name(col_name):
    col_name_str = str(col_name) # Always work with string representation

    # Check for the common pattern of stringified tuples of characters
    if col_name_str.startswith("('") and col_name_str.endswith("')") and ", '" in col_name_str:
        try:
            evaluated_literal = ast.literal_eval(col_name_str)
            # If it evaluates to a tuple and all its elements are strings, join them
            if isinstance(evaluated_literal, tuple) and all(isinstance(x, str) for x in evaluated_literal):
                return "".join(evaluated_literal)
        except (ValueError, SyntaxError):
            pass # Keep original string if literal_eval fails

    # Also handle actual tuple objects directly, in case they come from somewhere else
    if isinstance(col_name, tuple) and all(isinstance(x, str) for x in col_name):
        return "".join(col_name)

    # For anything else (already clean strings, or cases not handled above), return its string representation
    return col_name_str

# Apply the cleaning function to the columns of df_calc_mordred BEFORE merging
df_calc_mordred.columns = [clean_col_name(col) for col in df_calc_mordred.columns]

# Create a list of SMILES that correspond to the valid molecules for which Mordred descriptors were calculated.
# This list will have 194 entries, corresponding to the rows in df_calc_mordred.
valid_smiles_for_mordred = [smi for smi, mol in zip(smiles_list, mols) if mol is not None]

# Create a DataFrame for Mordred descriptors, adding the SMILES column for merging.
df_mordred_with_smiles = df_calc_mordred.copy()
df_mordred_with_smiles.insert(0, 'SMILES', valid_smiles_for_mordred)

# Crucial step: Remove duplicate SMILES from df_mordred_with_smiles to ensure unique merge keys.
# Since descriptors for the same SMILES should be identical, keeping the first occurrence is sufficient.
df_mordred_with_smiles_unique = df_mordred_with_smiles.drop_duplicates(subset=['SMILES'])

# Perform a left merge with the original df_classification_final (202 rows).
# This ensures that all original entries from df_classification_final are kept.
# For molecules where Mordred descriptors could not be calculated (due to invalid SMILES),
# the corresponding descriptor columns in df_class_mordred will be filled with NaN.
df_class_mordred = pd.merge(df_classification_final, df_mordred_with_smiles_unique, on='SMILES', how='left')

print(df_class_mordred.shape)

df_class_mordred

(202, 1829)


,SMILES,value,Experimental guidelines,ABC,ABCGG,nAcid,nBase,SpAbs_A,SpMax_A,SpDiam_A,...,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2
0,COc1nc(C)nc(NC(=O)NS(=O)(=O)c2ccccc2OCCCl)n1,Non-toxic,"OECD Guidelines 207: Earthworm, Acute Toxicity...",module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0.0,0.0,31.827086,2.404866,4.809731,...,10.047371,60.580733,401.056067,9.548954,1.797000e+03,38.0,128.0,145.0,9.979167,5.902778
1,CCO/N=C(\CC)C1=C(O)CC(c2c(C)cc(C)cc2C)CC1=O,Positive,"OECD Guidelines 207: Earthworm, Acute Toxicity...",module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0.0,0.0,29.074008,2.457016,4.914031,...,10.106469,58.483627,329.199094,6.454884,1.346000e+03,40.0,120.0,141.0,10.0,5.500000
2,CC1(C(=O)Nc2ccc(O)c(Cl)c2Cl)CCCCC1,Non-toxic,"OECD Guidelines 207: Earthworm, Acute Toxicity...",module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0.0,0.0,23.859449,2.402438,4.804875,...,9.932609,52.565569,301.063634,8.362879,7.060000e+02,31.0,98.0,115.0,7.618056,4.166667
3,O=C(NC(=O)c1c(F)cccc1F)Nc1cc(Cl)c(OC(F)(F)C(F)...,non-toxic,"OECD Guidelines 207: Earthworm, Acute Toxicity...",module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0.0,0.0,37.036801,2.42822,4.85644,...,10.445434,68.149026,509.978423,12.749461,3.276000e+03,53.0,166.0,192.0,15.236111,6.763889
4,O=[N+]([O-])c1cc(C(F)(F)F)c(Cl)c([N+](=O)[O-])...,negative,"OECD Guidelines 207: Earthworm, Acute Toxicity...",module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0.0,0.0,33.262277,2.50734,5.01468,...,10.467693,64.949854,463.951379,14.059133,2.126000e+03,51.0,154.0,181.0,14.486111,6.027778
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197,O=C(O)/C=C/C(=O)O.O=C(c1ccc(F)c(F)c1Nc1ccc(I)c...,positive,"OECD Guidelines 207: Earthworm, Acute Toxicity...",module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,2.0,2.0,multiple fragments (SpAbs_A/SpAbs),multiple fragments (SpMax_A/SpMax),multiple fragments (SpDiam_A/SpDiam),...,11.336272,108.272840,1178.137078,10.334536,1.380000e+11,105.0,358.0,422.0,25.569444,14.750000
198,O=C(c1ccc(F)c(F)c1Nc1ccc(I)cc1F)N1CC(O)([C@@H]...,positive,"OECD Guidelines 207: Earthworm, Acute Toxicity...",module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0.0,1.0,38.165569,2.525157,5.050314,...,10.615383,66.506898,531.063060,10.413001,2.520000e+03,50.0,164.0,197.0,10.423611,6.416667
199,Cc1c(F)cc(C(=O)NC2CC2)cc1-c1ccc(C(=O)NCC(C)(C)...,positive,"OECD Guidelines 207: Earthworm, Acute Toxicity...",module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0.0,0.0,35.006299,2.432869,4.858082,...,10.239281,81.589367,383.200905,7.096313,2.271000e+03,42.0,148.0,171.0,10.8125,5.930556
200,CClCCC=OF,negative,"OECD Guidelines 207: Earthworm, Acute Toxicity...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Finally, we want to use both sets of descriptors, so we will combine the two descriptor tables. As the descriptors were calculated using the same list of SMILES, their order should be identical, and a simple concatenation would suffice. However, we will use the merge method (see [documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)) as it offers two key advantages:

* It merges the tables using the SMILES column as a unique identifier. This helps prevent errors arising from changes in the order of the source tables and ensures that the data corresponds to the correct SMILES.

* It handles cases where columns exist in both DataFrames. In our case, we will add a suffix to indicate the source, allowing us to distinguish between versions.


In [ ]:
#@title Merge the two tables

#First, remove the "y" column from the Mordred table to avoid duplication of the endpoint.
# Note: The original df_class_mordred does not have a 'y' column, but rather a 'value' column. Assuming 'y' was a placeholder for the target variable column, I'll drop 'value' from df_class_mordred if it exists in both dataframes to avoid duplication.

df_class_mordred_no_target = df_class_mordred.drop(columns=['value', 'Experimental guidelines'], errors='ignore')

# Deduplicate df_class_rdkit before merging, similar to how df_mordred_with_smiles was handled.
df_class_rdkit_unique = df_class_rdkit.drop_duplicates(subset=['SMILES'])

#We merge the updated Mordred table (without the "y" column) with the RDKit table, using the "SMILES" column as the key and adding indicative suffixes
#To ensure consistency in the results and suffix labelling, it is important to use the RDKit DataFrame as the left table (i.e. before .merge)
df_class_all_descriptors=df_class_rdkit_unique.merge(df_class_mordred_no_target,on='SMILES',suffixes=('_rdk','_mdr'))

#Check the shapes of all three DataFrames to verify that the merge was successful
print(f'Shape of df_classification_final: {df_classification_final.shape}')
print(f'Shape of df_class_rdkit: {df_class_rdkit.shape}')
print(f'Shape of df_class_mordred: {df_class_mordred.shape}')
print(f'Shape of df_class_all_descriptors: {df_class_all_descriptors.shape}')

#Were any descriptors duplicated? Use this to check—specifically those with suffixes
#Be careful to exclude 'y_mdr' and 'y_rdk' from the check
print([ds for ds in df_class_all_descriptors.columns if 'mdr' in ds])
print([ds for ds in df_class_all_descriptors.columns if 'rdk' in ds])

Shape of df_classification_final: (202, 3)
Shape of df_class_rdkit: (202, 213)
Shape of df_class_mordred: (202, 1829)
Shape of df_class_all_descriptors: (202, 2039)
['BalabanJ_mdr', 'BertzCT_mdr', 'TPSA_mdr', 'LabuteASA_mdr', 'PEOE_VSA1_mdr', 'PEOE_VSA2_mdr', 'PEOE_VSA3_mdr', 'PEOE_VSA4_mdr', 'PEOE_VSA5_mdr', 'PEOE_VSA6_mdr', 'PEOE_VSA7_mdr', 'PEOE_VSA8_mdr', 'PEOE_VSA9_mdr', 'PEOE_VSA10_mdr', 'PEOE_VSA11_mdr', 'PEOE_VSA12_mdr', 'PEOE_VSA13_mdr', 'SMR_VSA1_mdr', 'SMR_VSA2_mdr', 'SMR_VSA3_mdr', 'SMR_VSA4_mdr', 'SMR_VSA5_mdr', 'SMR_VSA6_mdr', 'SMR_VSA7_mdr', 'SMR_VSA8_mdr', 'SMR_VSA9_mdr', 'SlogP_VSA1_mdr', 'SlogP_VSA2_mdr', 'SlogP_VSA3_mdr', 'SlogP_VSA4_mdr', 'SlogP_VSA5_mdr', 'SlogP_VSA6_mdr', 'SlogP_VSA7_mdr', 'SlogP_VSA8_mdr', 'SlogP_VSA9_mdr', 'SlogP_VSA10_mdr', 'SlogP_VSA11_mdr', 'EState_VSA1_mdr', 'EState_VSA2_mdr', 'EState_VSA3_mdr', 'EState_VSA4_mdr', 'EState_VSA5_mdr', 'EState_VSA6_mdr', 'EState_VSA7_mdr', 'EState_VSA8_mdr', 'EState_VSA9_mdr', 'EState_VSA10_mdr', 'VSA_EState1_m

###**Q3.** How many descriptors you calculate in total?

In [ ]:
print(len(df_class_all_descriptors.columns) - 3)

2036


Now, save this DataFrame as a CSV file named `Earthworm_acute_toxicity_descriptors.csv` in case it is needed for future steps or for inspection of the descriptor values.

In [ ]:
df_class_all_descriptors.to_csv(os.path.join(PATH, 'Earthworm_acute_toxicity_descriptors.csv'), index=False)

## Section 5: Pre-clearing of calculated descriptors

    

We will process this DataFrame in order to:

* Eliminate descriptor values that are strings (replace with a non-numerical value, i.e. `np.nan`)

* Remove extremely large or infinite values (also replace with `np.nan`)

* Filter out columns with a high proportion of missing values (greater than 15%)

To achieve this, we will first identify the columns containing descriptors and then apply the transformations specifically to those columns.

In [ ]:
#@title Read the file
# To avoid using an incorrect version of the DataFrame or propagating errors from previous sections,
# it is advisable to read the descriptors file again and create a new DataFrame.

df_classification_with_descriptors = pd.read_csv(os.path.join(PATH, 'Earthworm_acute_toxicity_descriptors.csv'))
print(df_classification_with_descriptors.shape)

(202, 2039)


The pre-clearing process is applied only to the descriptors, so we need a list of descriptor column names. In theory, these should include all columns in the DataFrame except the first two (`SMILES` and `y`).


In [ ]:
#@title Select the descriptor columns
# Print all columns beforehand (for verification)
print(df_classification_with_descriptors.columns)
# desc_columns should include all columns except the first three: 'SMILES', 'value', and 'Experimental guidelines'.
desc_columns = df_classification_with_descriptors.columns[3:]
# Print desc_columns (for verification)
print(desc_columns)

Index(['SMILES', 'value', 'Experimental guidelines', 'MaxAbsEStateIndex',
       'MaxEStateIndex', 'MinAbsEStateIndex', 'MinEStateIndex', 'qed', 'SPS',
       'MolWt',
       ...
       'SRW10', 'TSRW10', 'MW', 'AMW', 'WPath', 'WPol', 'Zagreb1', 'Zagreb2',
       'mZagreb1', 'mZagreb2'],
      dtype='str', length=2039)
Index(['MaxAbsEStateIndex', 'MaxEStateIndex', 'MinAbsEStateIndex',
       'MinEStateIndex', 'qed', 'SPS', 'MolWt', 'HeavyAtomMolWt', 'ExactMolWt',
       'NumValenceElectrons',
       ...
       'SRW10', 'TSRW10', 'MW', 'AMW', 'WPath', 'WPol', 'Zagreb1', 'Zagreb2',
       'mZagreb1', 'mZagreb2'],
      dtype='str', length=2036)


Next, we will filter out those columns with a high proportion of missing values (greater than 15%).

In [ ]:
#@title Remove columns with an excessive number of missing values (NaN)
# Remove columns in which all values are np.nan
print('Shape before removing:', df_classification_with_descriptors.shape)
df_classification_with_descriptors = df_classification_with_descriptors.dropna(axis = 1, how = 'all')
print('Shape after removing all nule descriptors:', df_classification_with_descriptors.shape)
nan_limit=0.15
all_nan=df_classification_with_descriptors.columns[df_classification_with_descriptors.isna().sum()>len(df_classification_with_descriptors.isna())*nan_limit].tolist()

out_columns = list(df_classification_with_descriptors.columns)

if len(all_nan)>0:
    print('Warning! Some of descriptors (',len(all_nan),') have too many NaN values and have been removed:', all_nan, 'The limit is marked to the ', nan_limit*100, "%")
for i in all_nan:
    out_columns.remove(i)

df_classification_with_descriptors = df_classification_with_descriptors[out_columns]
print('Shape after removing if more than 15% empty:',df_classification_with_descriptors.shape)

Shape before removing: (202, 1630)
Shape after removing all nule descriptors: (202, 1630)
Shape after removing if more than 15% empty: (202, 1630)


Finally, we will save the pre-cleaned descriptor matrix as a CSV file named `Earthworm_acute_toxicity_descriptors_cleaned.csv' . Please ensure that this file is saved, as it will be required in the next step.